In [1]:
!pip install datasets tiktoken -q

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import time
import tiktoken
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader


In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == "cuda":
  print(f"GPU: {torch.cuda.get_device_name(0)}")
  print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

GPU: NVIDIA A100-SXM4-80GB
VRAM: 79.3 GB


In [4]:
dataset = load_dataset("wikitext", "wikitext-103-raw-v1")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [5]:
print(f"Train: {len(dataset['train'])} lines")
print(f"Val:   {len(dataset['validation'])} lines")
print(f"Test:  {len(dataset['test'])} lines")

# Look at a sample
print(f"\nSample text:")
print(dataset['train'][100]['text'][:500])

Train: 1801350 lines
Val:   3760 lines
Test:  4358 lines

Sample text:
 96 ammunition packing boxes 



In [6]:
enc = tiktoken.get_encoding("gpt2")

In [7]:
sample = "The capital of France is Paris."
tokens = enc.encode(sample)
print(f"Text:    {sample}")
print(f"Tokens:  {tokens}")

Text:    The capital of France is Paris.
Tokens:  [464, 3139, 286, 4881, 318, 6342, 13]


In [8]:
print(f"Decoded: {enc.decode(tokens)}")
print(f"Token count: {len(tokens)}")

Decoded: The capital of France is Paris.
Token count: 7


In [9]:
print(f"\nToken breakdown:")
for tok_id in tokens:
    print(f"  {tok_id:6d} → '{enc.decode([tok_id])}'")


Token breakdown:
     464 → 'The'
    3139 → ' capital'
     286 → ' of'
    4881 → ' France'
     318 → ' is'
    6342 → ' Paris'
      13 → '.'


In [10]:
VOCAB_SIZE = enc.n_vocab
print(f"\nVocab size: {VOCAB_SIZE}")


Vocab size: 50257


In [11]:
def tokenize_dataset(dataset_split):
  all_tokens = []
  for example in dataset_split:
    text = example["text"]
    if text.strip():
      tokens = enc.encode(text)
      all_tokens.extend(tokens)
  return all_tokens

In [12]:
from pygments import token
print("Tokenizing train set...")
train_tokens = tokenize_dataset(dataset["train"])
print(f"Train tokens: {len(train_tokens):,}")

print("Tokenizing val set...")
val_tokens = tokenize_dataset(dataset["validation"])
print(f"Val tokens:   {len(val_tokens):,}")

print(f"\nFirst 20 tokens: {train_tokens[:20]}")
print(f"Decoded: {enc.decode(train_tokens[:20])}")

Tokenizing train set...
Train tokens: 117,920,140
Tokenizing val set...
Val tokens:   247,289

First 20 tokens: [796, 569, 18354, 7496, 17740, 6711, 796, 220, 198, 2311, 73, 13090, 645, 569, 18354, 7496, 513, 1058, 791, 47398]
Decoded:  = Valkyria Chronicles III = 
 Senjō no Valkyria 3 : Unrecorded


In [13]:
class TextDataset(Dataset):
  def __init__(self, tokens, seq_len):
    self.tokens = tokens
    self.seq_len = seq_len
    self.num_samples = (len(tokens)-1) // seq_len

  def __len__(self):
    return self.num_samples

  def __getitem__(self, idx):
    start = self.seq_len*idx
    x = self.tokens[start:start+self.seq_len]
    y = self.tokens[start+1:start+self.seq_len+1]
    return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)



In [14]:
SEQ_LEN = 256
BATCH_SIZE = 64

train_dataset = TextDataset(train_tokens, SEQ_LEN)
val_dataset = TextDataset(val_tokens, SEQ_LEN)

In [15]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [16]:
print(f"Sequence length: {SEQ_LEN}")
print(f"Train samples: {len(train_dataset):,}")
print(f"Val samples:   {len(val_dataset):,}")

x, y = train_dataset[0]
print(f"\nInput shape:  {x.shape}")
print(f"Target shape: {y.shape}")
print(f"Input:  {x[:10]}")
print(f"Target: {y[:10]}")
print(f"\nInput decoded:  {enc.decode(x[:10].tolist())}")
print(f"Target decoded: {enc.decode(y[:10].tolist())}")

Sequence length: 256
Train samples: 460,625
Val samples:   965

Input shape:  torch.Size([256])
Target shape: torch.Size([256])
Input:  tensor([  796,   569, 18354,  7496, 17740,  6711,   796,   220,   198,  2311])
Target: tensor([  569, 18354,  7496, 17740,  6711,   796,   220,   198,  2311,    73])

Input decoded:   = Valkyria Chronicles III = 
 Sen
Target decoded:  Valkyria Chronicles III = 
 Senj


In [17]:
class TokenEmbedding(nn.Module):
  def __init__(self, vocab_size, d_model):
    super().__init__()
    self.d_model = d_model
    self.embedding = nn.Embedding(vocab_size, d_model)

  def forward(self, x):
    return self.embedding(x) * math.sqrt(self.d_model)

In [18]:
test_embed = TokenEmbedding(vocab_size = VOCAB_SIZE, d_model=512)
dummy = torch.tensor([[1, 21, 91, 2]])
output = test_embed(dummy)
print(f"Input shape:  {dummy.shape}")    # (1, 4)
print(f"Output shape: {output.shape}")   # (1, 4, 512)
print(f"Vocab size: {VOCAB_SIZE}")

Input shape:  torch.Size([1, 4])
Output shape: torch.Size([1, 4, 512])
Vocab size: 50257


In [19]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len=5000, dropout=0.1):
    super().__init__()
    self.dropout = nn.Dropout(dropout)
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000)/d_model))
    pe[:, 0::2] = torch.sin(position*div_term)
    pe[:, 1::2] = torch.cos(position*div_term)
    pe = pe.unsqueeze(0)
    self.register_buffer('pe', pe)

  def forward(self, x):
    x = x + self.pe[:, :x.size(1), :]
    return self.dropout(x)



In [20]:
test_pe = PositionalEncoding(d_model=512)
dummy = torch.zeros(2,10,512)
output = test_pe(dummy)
print(f"Input shape:  {dummy.shape}")    # (2, 10, 512)
print(f"Output shape: {output.shape}")   # (2, 10, 512)

Input shape:  torch.Size([2, 10, 512])
Output shape: torch.Size([2, 10, 512])


In [21]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, n_heads):
    super().__init__()
    assert d_model % n_heads == 0
    self.d_model = d_model
    self.n_heads = n_heads
    self.d_k = d_model // n_heads
    self.W_q = nn.Linear(d_model, d_model)
    self.W_k = nn.Linear(d_model, d_model)
    self.W_v = nn.Linear(d_model, d_model)
    self.W_o = nn.Linear(d_model, d_model)

  def scaled_dot_product_attention(self, Q, K, V, mask=None):
    scores = torch.matmul(Q, K.transpose(-2,-1))/math.sqrt(self.d_k)
    if mask is not None:
      scores = scores.masked_fill(mask==0, float('-inf'))
    attn_weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output

  def forward(self, query, key, value, mask=None):
    batch_size = query.size(0)
    Q = self.W_q(query)
    K = self.W_k(key)
    V = self.W_v(value)

    Q = Q.view(batch_size, -1, self.n_heads, self.d_k).transpose(1,2)
    K = K.view(batch_size, -1, self.n_heads, self.d_k).transpose(1,2)
    V = V.view(batch_size, -1, self.n_heads, self.d_k).transpose(1,2)
    attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
    attn_output = attn_output.transpose(1,2).contiguous().view(batch_size, -1, self.d_model)
    output = self.W_o(attn_output)
    return output



In [22]:
test_mha = MultiHeadAttention(d_model=512, n_heads=8)
dummy = torch.randn(2, 10, 512)
output = test_mha(dummy, dummy, dummy)
print(f"Input shape:  {dummy.shape}")    # (2, 10, 512)
print(f"Output shape: {output.shape}")   # (2, 10, 512)

Input shape:  torch.Size([2, 10, 512])
Output shape: torch.Size([2, 10, 512])


In [23]:
class FeedForward(nn.Module):
  def __init__(self, d_model, d_ff, dropout=0.1):
    super().__init__()
    self.linear1 = nn.Linear(d_model, d_ff)
    self.linear2 = nn.Linear(d_ff, d_model)
    self.dropout = nn.Dropout(dropout)
    self.gelu = nn.GELU()

  def forward(self, x):
    x = self.linear1(x)
    x = self.gelu(x)
    x = self.dropout(x)
    x = self.linear2(x)
    return x




In [24]:
test_ff = FeedForward(d_model=512, d_ff=2048)
dummy = torch.randn(2,10,512)
output = test_ff(dummy)
print(f"Input shape:  {dummy.shape}")    # (2, 10, 512)
print(f"Output shape: {output.shape}")   # (2, 10, 512)

Input shape:  torch.Size([2, 10, 512])
Output shape: torch.Size([2, 10, 512])


In [25]:
class DecoderLayer(nn.Module):
  def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
    super().__init__()
    self.self_attention = MultiHeadAttention(d_model=d_model, n_heads=n_heads)
    self.feed_forward = FeedForward(d_model=d_model, d_ff=d_ff, dropout=dropout)
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.dropout1 = nn.Dropout(dropout)
    self.dropout2 = nn.Dropout(dropout)

  def forward(self, x, mask):
    attn_output = self.self_attention(x, x, x, mask)
    attn_output = self.dropout1(attn_output)
    x = self.norm1(x + attn_output)
    ff_output = self.feed_forward(x)
    ff_output = self.dropout2(ff_output)
    x = self.norm2(x + ff_output)
    return x



In [26]:
test_layer = DecoderLayer(d_model=512, n_heads=8, d_ff=2048)
dummy = torch.randn(2, 10, 512)
output = test_layer(dummy, mask=None)
print(f"Input shape:  {dummy.shape}")    # (2, 10, 512)
print(f"Output shape: {output.shape}")   # (2, 10, 512)

Input shape:  torch.Size([2, 10, 512])
Output shape: torch.Size([2, 10, 512])


In [27]:
class GPTModel(nn.Module):
  def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq_len, dropout=0.1):
    super().__init__()
    self.token_embeddings = TokenEmbedding(vocab_size=vocab_size, d_model=d_model)
    self.positional_encoding = PositionalEncoding(d_model=d_model, max_len=max_seq_len, dropout=dropout)
    self.layers = nn.ModuleList([DecoderLayer(d_model=d_model, n_heads=n_heads, d_ff=d_ff, dropout=dropout) for _ in range(n_layers)])
    self.final_norm = nn.LayerNorm(d_model)
    self.fc_out = nn.Linear(d_model, vocab_size)

  def forward(self, x, mask=None):
    x = self.token_embeddings(x)
    x = self.positional_encoding(x)
    for layer in self.layers:
      x = layer(x, mask)
    x = self.final_norm(x)
    x = self.fc_out(x)
    return x


In [28]:
D_MODEL = 512
N_HEADS = 8
D_FF = 2048
N_LAYERS = 6
MAX_SEQ_LEN = SEQ_LEN  # 256
DROPOUT = 0.1

In [29]:
device

device(type='cuda')

In [30]:
model = GPTModel(vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF, n_layers=N_LAYERS, max_seq_len=MAX_SEQ_LEN, dropout=DROPOUT)

In [31]:
dummy = torch.randint(0, VOCAB_SIZE, (2, 10)).to(device)
model.to(device) # Move the model to the device
output = model(dummy)
print(f"Input shape:  {dummy.shape}")     # (2, 10)
print(f"Output shape: {output.shape}")    # (2, 10, 50257)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters:   {total_params:,}")

Input shape:  torch.Size([2, 10])
Output shape: torch.Size([2, 10, 50257])
Parameters:   70,428,753


In [32]:
dummy = torch.randint(0, VOCAB_SIZE, (2, 10)).to(device)
output = model(dummy)
print(f"Output shape: {output.shape}")
print(f"Expected last dim: {VOCAB_SIZE}")
print(f"Actual last dim: {output.shape[-1]}")

Output shape: torch.Size([2, 10, 50257])
Expected last dim: 50257
Actual last dim: 50257


In [33]:
t1 = torch.ones(3,3)
t2 = torch.randn(3,3)

In [34]:
torch.tril(t1).bool().unsqueeze(0).unsqueeze(1)

tensor([[[[ True, False, False],
          [ True,  True, False],
          [ True,  True,  True]]]])

In [35]:
torch.tril(t1).bool().unsqueeze(0).unsqueeze(0)

tensor([[[[ True, False, False],
          [ True,  True, False],
          [ True,  True,  True]]]])

In [36]:
torch.tril(t2).bool()

tensor([[ True, False, False],
        [ True,  True, False],
        [ True,  True,  True]])

In [37]:
def create_causal_mask(seq_len, device):
  mask = torch.tril(torch.ones(seq_len, seq_len, device=device)).bool()
  mask = mask.unsqueeze(0).unsqueeze(1)
  return mask

In [38]:
test_mask = create_causal_mask(5, 'cpu')
print(f"Mask shape: {test_mask.shape}")
print(f"Mask:\n{test_mask.squeeze()}")

Mask shape: torch.Size([1, 1, 5, 5])
Mask:
tensor([[ True, False, False, False, False],
        [ True,  True, False, False, False],
        [ True,  True,  True, False, False],
        [ True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True]])


In [39]:
test_mask.squeeze(0).squeeze(0)

tensor([[ True, False, False, False, False],
        [ True,  True, False, False, False],
        [ True,  True,  True, False, False],
        [ True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True]])

In [40]:
test_mask.squeeze(0).squeeze(1)

tensor([[[ True, False, False, False, False],
         [ True,  True, False, False, False],
         [ True,  True,  True, False, False],
         [ True,  True,  True,  True, False],
         [ True,  True,  True,  True,  True]]])

In [41]:
LEARNING_RATE = 3e-4
EPOCHS = 5

In [42]:
model.parameters()

<generator object Module.parameters at 0x799e7343e960>

In [43]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.98, 0.98), eps=1e-9)

In [44]:
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {device}")


Model parameters: 70,428,753
Device: cuda


In [45]:
def train_one_epoch(model, dataloader, optimizer, loss_fn, device, seq_len):
  model.train()
  total_loss = 0
  total_tokens = 0
  start_time = time.time()
  mask = create_causal_mask(seq_len, device)
  for batch_idx, (x,y) in enumerate(dataloader):
    x = x.to(device)
    y = y.to(device)
    output = model(x, mask)
    output = output.reshape(-1, output.size(-1))
    y = y.reshape(-1)
    loss = loss_fn(output, y)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    batch_tokens = x.size(0) * x.size(1)
    total_loss += loss.item() * batch_tokens
    total_tokens += batch_tokens

    if (batch_idx+1) % 500 == 0:
      elapsed = time.time() - start_time
      avg_loss = total_loss/total_tokens
      print(f"  Batch {batch_idx+1}/{len(dataloader)} | Loss: {avg_loss:.4f} | Time: {elapsed:.0f}s")

  return total_loss/total_tokens



In [46]:
def evaluate(model, dataloader, loss_fn, device, seq_len):
  model.eval()
  total_loss = 0
  total_tokens = 0
  mask = create_causal_mask(seq_len, device)
  with torch.no_grad():
    for x,y in dataloader:
      x = x.to(device)
      y = y.to(device)
      output = model(x, mask)
      output = output.reshape(-1, output.size(-1))
      y = y.reshape(-1)
      loss = loss_fn(output, y)
      batch_tokens = x.size(0) * x.size(1)
      total_loss += loss.item() * batch_tokens
      total_tokens += batch_tokens
  return total_loss/total_tokens





In [47]:
for epoch in range(EPOCHS):
  start_time = time.time()
  train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device, SEQ_LEN)
  val_loss = evaluate(model, val_loader, loss_fn, device, SEQ_LEN)
  elapsed = time.time() - start_time
  train_ppl = math.exp(train_loss)
  val_ppl = math.exp(val_loss)
  print(f"Epoch {epoch+1}/{EPOCHS} | "
      f"Train Loss: {train_loss:.4f} | "
      f"Val Loss: {val_loss:.4f} | "
      f"Train PPL: {train_ppl:.2f} | "
      f"Val PPL: {val_ppl:.2f} | "
      f"Time: {elapsed:.0f}s")


  Batch 500/7198 | Loss: 6.4388 | Time: 156s
  Batch 1000/7198 | Loss: 6.0084 | Time: 312s
  Batch 1500/7198 | Loss: 5.7385 | Time: 468s
  Batch 2000/7198 | Loss: 5.5437 | Time: 624s
  Batch 2500/7198 | Loss: 5.3973 | Time: 781s
  Batch 3000/7198 | Loss: 5.2798 | Time: 937s
  Batch 3500/7198 | Loss: 5.1820 | Time: 1093s
  Batch 4000/7198 | Loss: 5.0997 | Time: 1249s
  Batch 4500/7198 | Loss: 5.0285 | Time: 1405s
  Batch 5000/7198 | Loss: 4.9666 | Time: 1561s
  Batch 5500/7198 | Loss: 4.9111 | Time: 1717s
  Batch 6000/7198 | Loss: 4.8610 | Time: 1873s
  Batch 6500/7198 | Loss: 4.8160 | Time: 2029s
  Batch 7000/7198 | Loss: 4.7745 | Time: 2185s
Epoch 1/5 | Train Loss: 4.7591 | Val Loss: 4.0424 | Train PPL: 116.64 | Val PPL: 56.96 | Time: 2249s
  Batch 500/7198 | Loss: 4.1564 | Time: 156s
  Batch 1000/7198 | Loss: 4.1444 | Time: 312s
  Batch 1500/7198 | Loss: 4.1362 | Time: 468s
  Batch 2000/7198 | Loss: 4.1268 | Time: 624s
  Batch 2500/7198 | Loss: 4.1172 | Time: 780s
  Batch 3000/7198 |

In [51]:
enc.encode("I am a student")

[40, 716, 257, 3710]

In [49]:
import json
import os

os.makedirs("gpt_model", exist_ok=True)

# 1. Save model weights
torch.save(model.state_dict(), "gpt_model/model.pt")

# 2. Save hyperparameters
config = {
    "vocab_size": VOCAB_SIZE,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "d_ff": D_FF,
    "n_layers": N_LAYERS,
    "max_seq_len": MAX_SEQ_LEN,
    "dropout": DROPOUT
}

with open("gpt_model/config.json", "w") as f:
    json.dump(config, f)

print("Saved files:")
for f in os.listdir("gpt_model"):
    size = os.path.getsize(f"gpt_model/{f}") / (1024*1024)
    print(f"  {f} — {size:.2f} MB")

Saved files:
  config.json — 0.00 MB
  model.pt — 269.19 MB


In [53]:
t1 = torch.randn(1,4,10)

In [55]:
t1

tensor([[[-0.3263, -1.2019,  0.7890,  0.6385, -2.0553, -0.3794, -0.2221,
          -0.1057,  0.1537,  0.7709],
         [ 1.5364, -0.3426,  0.0660, -0.8048,  0.4095,  1.4100,  1.9833,
          -1.5596,  1.9374, -0.1343],
         [ 0.2398,  0.6132, -0.4167, -0.4246, -1.0030, -0.3911,  1.3627,
           0.1310,  0.4185,  0.8121],
         [ 0.9758,  0.2129,  0.6353,  0.8077,  1.1855,  0.4917,  0.3925,
          -0.9562,  1.2356, -0.6677]]])

In [57]:
t1[0,-1,:]

tensor([ 0.9758,  0.2129,  0.6353,  0.8077,  1.1855,  0.4917,  0.3925, -0.9562,
         1.2356, -0.6677])

In [62]:
def generate(model, prompt, max_new_tokens=100, temperature=0.8):
  model.eval()
  token_ids = enc.encode(prompt)
  with torch.no_grad():
    for _ in range(max_new_tokens):
      input_ids = token_ids[-SEQ_LEN:]
      x = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)
      mask = create_causal_mask(x.size(1), device)
      output = model(x, mask)
      logits = output[0, -1, :]
      logits = logits/temperature
      probs = torch.softmax(logits, dim=-1)
      next_token = torch.multinomial(probs, num_samples=1).item()
      token_ids.append(next_token)
  generated_text = enc.decode(token_ids)
  return generated_text



In [63]:
prompts = [
    "The capital of France is",
    "In the year 1969, the first",
    "The theory of relativity states that",
    "Football is a sport that",
]

print("=== Generated Text ===\n")
for prompt in prompts:
    generated = generate(model, prompt, max_new_tokens=80, temperature=0.8)
    print(f"Prompt: {prompt}")
    print(f"Generated: {generated}")
    print("-" * 80)
    print()

=== Generated Text ===

Prompt: The capital of France is
Generated: The capital of France is home to one of the world 's largest cities . 
 The city 's oldest entry , the Port Authority ( RAAF ) , was established in 1983 . In the early 1980s , the city was given the title " Republic of Ireland Under @-@ 18 " by the Queen 's Council , which is followed by the Lord Mayor 's Office and King 's Office . The city
--------------------------------------------------------------------------------

Prompt: In the year 1969, the first
Generated: In the year 1969, the first @-@ ever national level . The highest amount of Early Cold War radar in the world is 1 @,@ 100 m ( 6 @,@ 600 ft ) , which is 67 metres ( 318 ft ) above sea level . 
 The oldest motorway on the North Coast , which is served by of the Pennsylvania Department of Transportation ( MDOT ) ( MDOT ) and the Baltimore County Council
--------------------------------------------------------------------------------

Prompt: The theory of r

In [64]:
app_code = '''import torch
import torch.nn as nn
import math
import json
import tiktoken
import gradio as gr

# ============================================================
# Model Architecture
# ============================================================

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, V)
        return output

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)
        Q = Q.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        output = self.W_o(attn_output)
        return output

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.gelu = nn.GELU()

    def forward(self, x):
        x = self.linear1(x)
        x = self.gelu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn_output = self.self_attention(x, x, x, mask)
        attn_output = self.dropout1(attn_output)
        x = self.norm1(x + attn_output)
        ff_output = self.feed_forward(x)
        ff_output = self.dropout2(ff_output)
        x = self.norm2(x + ff_output)
        return x

class GPTModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq_len, dropout=0.1):
        super().__init__()
        self.token_embedding = TokenEmbedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len=max_seq_len, dropout=dropout)
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x, mask=None):
        x = self.token_embedding(x)
        x = self.positional_encoding(x)
        for layer in self.layers:
            x = layer(x, mask)
        x = self.final_norm(x)
        x = self.fc_out(x)
        return x

# ============================================================
# Masking
# ============================================================

def create_causal_mask(seq_len, device):
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device)).bool()
    mask = mask.unsqueeze(0).unsqueeze(1)
    return mask

# ============================================================
# Load Model
# ============================================================

device = torch.device('cpu')

with open("config.json", "r") as f:
    config = json.load(f)

enc = tiktoken.get_encoding("gpt2")

model = GPTModel(
    vocab_size=config["vocab_size"],
    d_model=config["d_model"],
    n_heads=config["n_heads"],
    d_ff=config["d_ff"],
    n_layers=config["n_layers"],
    max_seq_len=config["max_seq_len"],
    dropout=config["dropout"]
)

model.load_state_dict(torch.load("model.pt", map_location=device))
model.eval()

# ============================================================
# Generation Function
# ============================================================

def generate(prompt, max_new_tokens=100, temperature=0.8):
    if not prompt.strip():
        return "Please enter a prompt."

    token_ids = enc.encode(prompt)
    max_seq_len = config["max_seq_len"]

    with torch.no_grad():
        for _ in range(int(max_new_tokens)):
            input_ids = token_ids[-max_seq_len:]
            x = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)
            mask = create_causal_mask(x.size(1), device)
            output = model(x, mask)
            logits = output[0, -1, :]
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
            token_ids.append(next_token)

    generated_text = enc.decode(token_ids)

    # Clean up WikiText artifacts
    generated_text = generated_text.replace(" @-@ ", "-")
    generated_text = generated_text.replace(" @,@ ", ",")
    generated_text = generated_text.replace(" @.@ ", ".")

    return generated_text

# ============================================================
# Gradio Interface
# ============================================================

examples = [
    ["The capital of France is", 80, 0.8],
    ["In the year 1969, the first", 80, 0.8],
    ["The theory of relativity states that", 100, 0.7],
    ["Football is a sport that", 80, 0.9],
    ["The history of artificial intelligence", 100, 0.8],
]

demo = gr.Interface(
    fn=generate,
    inputs=[
        gr.Textbox(label="Prompt", placeholder="Enter a text prompt..."),
        gr.Slider(minimum=10, maximum=200, value=80, step=10, label="Max New Tokens"),
        gr.Slider(minimum=0.1, maximum=2.0, value=0.8, step=0.1, label="Temperature"),
    ],
    outputs=gr.Textbox(label="Generated Text"),
    title="GPT Text Generator",
    description="Decoder-only Transformer built from scratch using PyTorch. Trained on WikiText-103 (~118M tokens from Wikipedia). Architecture: 6 decoder layers, 8 attention heads, d_model=512, ~70M parameters. The model generates text by predicting one token at a time.",
    examples=examples,
    theme=gr.themes.Soft()
)

if __name__ == "__main__":
    demo.launch()
'''

with open("gpt_model/app.py", "w") as f:
    f.write(app_code)

# Requirements
with open("gpt_model/requirements.txt", "w") as f:
    f.write("torch\ntiktoken\naudioop-lts\n")

# README
readme = """---
title: GPT Text Generator
emoji: 🤖
colorFrom: purple
colorTo: blue
sdk: gradio
sdk_version: "5.12.0"
app_file: app.py
pinned: false
---

# GPT Text Generator

A decoder-only Transformer built entirely from scratch using PyTorch, trained on the WikiText-103 dataset.

## Architecture
- **Type:** Decoder-only Transformer (GPT-style)
- **Decoder Layers:** 6
- **Attention Heads:** 8
- **Model Dimension:** 512
- **Feed-Forward Dimension:** 2048
- **Parameters:** ~70M
- **Tokenizer:** GPT-2 BPE (tiktoken)
- **Dataset:** WikiText-103 (~118M tokens from Wikipedia)
- **Validation Perplexity:** 36.06

## How It Works
Every component — embeddings, positional encoding, multi-head causal attention, feed-forward layers — was implemented from scratch. No pre-trained weights or HuggingFace model classes were used. The model predicts the next token autoregressively using causal masking.

## Controls
- **Max New Tokens:** Controls how many tokens the model generates (10-200)
- **Temperature:** Controls randomness. Low (0.1) = repetitive but safe. High (2.0) = creative but chaotic. Default 0.8 works well.

## Limitations
- Trained on Wikipedia text only, so outputs have Wikipedia-style writing
- 70M parameters is small — factual accuracy is limited
- No instruction tuning — the model continues text rather than answering questions
"""

with open("gpt_model/README.md", "w") as f:
    f.write(readme)

print("All files created!")
for f in os.listdir("gpt_model"):
    size = os.path.getsize(f"gpt_model/{f}") / (1024*1024)
    print(f"  {f} — {size:.2f} MB")

All files created!
  requirements.txt — 0.00 MB
  app.py — 0.01 MB
  README.md — 0.00 MB
  config.json — 0.00 MB
  model.pt — 269.19 MB
